# Aula 02: jailbreaks manuais no DeepSeek-R1

PCS5917 IA Adversarial, EP-USP

Ataques manuais de jailbreak contra o [`deepseek-ai/DeepSeek-R1`](https://huggingface.co/deepseek-ai/DeepSeek-R1),
chamado pelo Hugging Face Inference Providers com o provider Novita. As técnicas vêm de um dataset de prompts de
jailbreak coletados na internet. Rodei cada ataque em inglês e em português, em células separadas, para comparar.

Uso acadêmico, em ambiente controlado.

## 1. Setup

In [ ]:
!pip install -q -U huggingface_hub datasets pandas

In [ ]:
import os, re, datetime
import pandas as pd
from huggingface_hub import InferenceClient

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass  # rodando local, com HF_TOKEN no ambiente

MODEL = "deepseek-ai/DeepSeek-R1"
PROVIDER = "novita"

client = InferenceClient(provider=PROVIDER, api_key=os.environ["HF_TOKEN"])

### Funções auxiliares

O R1 devolve o raciocínio entre `<think>...</think>` antes da resposta. Na Novita a tag de fechamento às vezes
não vem, e parte do texto cai no campo `reasoning_content`, então `split_think` trata os dois casos.
`ask` guarda cada chamada em `RESULTS` e `judge` anota se o modelo recusou, respondeu em parte ou foi quebrado.

In [ ]:
RESULTS = []

def split_think(content, reasoning_field=None):
    content = content or ""
    if "</think>" in content:
        head, answer = content.split("</think>", 1)
        return head.replace("<think>", "").strip(), answer.strip()
    if content.lstrip().startswith("<think>"):
        # sem </think>: devolve tudo junto
        return (content + (reasoning_field or "")).replace("<think>", "").strip(), ""
    return (reasoning_field or "").strip(), content.strip()

def ask(prompt, system=None, attack="", lang="", history=None, max_tokens=2048, show_reasoning=True):
    messages = [{"role": "system", "content": system}] if system else []
    messages += history or []
    messages.append({"role": "user", "content": prompt})

    completion = client.chat.completions.create(
        model=MODEL, messages=messages, max_tokens=max_tokens, temperature=0.6,
    )
    msg = completion.choices[0].message
    reasoning, answer = split_think(msg.content, getattr(msg, "reasoning_content", None))

    print(f"=== [{attack} | {lang}] PROMPT ===\n{prompt}\n")
    if show_reasoning and reasoning:
        print("=== RACIOCÍNIO ===\n" + reasoning[:3000] + ("\n[...]" if len(reasoning) > 3000 else "") + "\n")
    print("=== RESPOSTA ===\n" + (answer or "(vazia, ver raciocínio)"))

    RESULTS.append({"attack": attack, "lang": lang, "prompt": prompt, "system": system,
                    "reasoning": reasoning, "answer": answer, "outcome": None, "note": "",
                    "timestamp": datetime.datetime.now().isoformat()})
    return answer

def judge(outcome, note=""):
    # recusou / parcial / jailbreak
    RESULTS[-1]["outcome"], RESULTS[-1]["note"] = outcome, note

## 2. Teste de conexão

In [ ]:
ask("Hello! In one sentence, who are you?", attack="sanity", lang="EN")

In [ ]:
ask("Olá! Em uma frase, quem é você?", attack="sanity", lang="PT")

## 3. Dataset

[`TrustAIRLab/in-the-wild-jailbreak-prompts`](https://huggingface.co/datasets/TrustAIRLab/in-the-wild-jailbreak-prompts),
de Shen et al. (CCS 2024): prompts de jailbreak coletados no Reddit, Discord e sites de prompts.

In [ ]:
from datasets import load_dataset

DATASET_ID = "TrustAIRLab/in-the-wild-jailbreak-prompts"
CONFIG = "jailbreak_2023_12_25"
SPLIT = "train"
TEXT_COLUMN = "prompt"

df_ds = load_dataset(DATASET_ID, CONFIG, split=SPLIT).to_pandas()
print(df_ds.shape)
df_ds.head()

In [ ]:
for t in df_ds[TEXT_COLUMN].sample(5, random_state=42):
    print(t[:600].replace("\n", " "), "\n---")

Busca por palavras-chave para separar os prompts por técnica.

In [ ]:
KEYWORDS = {
    "A1": r"\bDAN\b|\bAIM\b|developer mode|you are now|act as",
    "A2": r"hypothetical|fiction|story|novel|screenplay|imagine",
    "A3": r"never refuse|do not refuse|start your (response|answer) with|without any warnings",
    "A4": r"base64|leetspeak|encode|cipher|reverse the",
    "A5": r"ignore (all )?(previous|prior) instructions|system prompt|reveal your instructions",
}

def exemplos(tecnica, n=3, maxlen=800):
    hits = df_ds[df_ds[TEXT_COLUMN].str.contains(KEYWORDS[tecnica], case=False, regex=True, na=False)]
    print(f"{tecnica}: {len(hits)} prompts\n")
    for t in hits[TEXT_COLUMN].sample(min(n, len(hits)), random_state=0):
        print(t[:maxlen], "\n" + "-" * 80)

for k in KEYWORDS:
    exemplos(k, n=2)

## 4. Ataques

Mesmo pedido-alvo em todas as técnicas, para dar para comparar com o baseline. A versão em português é a
tradução do mesmo prompt.

### A0: Baseline

Pedido direto, sem técnica nenhuma.

#### A0: Inglês

In [ ]:
PROMPT = """
"""
SYSTEM = None
ask(PROMPT, system=SYSTEM, attack="A0 Baseline", lang="EN")

In [ ]:
judge("", note="")

#### A0: Português

In [ ]:
PROMPT = """
"""
SYSTEM = None
ask(PROMPT, system=SYSTEM, attack="A0 Baseline", lang="PT")

In [ ]:
judge("", note="")

### A1: Persona

Personas no estilo DAN e AIM, as mais comuns no dataset.

#### A1: Inglês

In [ ]:
PROMPT = """
"""
SYSTEM = None
ask(PROMPT, system=SYSTEM, attack="A1 Persona", lang="EN")

In [ ]:
judge("", note="")

#### A1: Português

In [ ]:
PROMPT = """
"""
SYSTEM = None
ask(PROMPT, system=SYSTEM, attack="A1 Persona", lang="PT")

In [ ]:
judge("", note="")

### A2: Cenário fictício

O pedido vai dentro de uma história ou roteiro.

#### A2: Inglês

In [ ]:
PROMPT = """
"""
SYSTEM = None
ask(PROMPT, system=SYSTEM, attack="A2 Cenário fictício", lang="EN")

In [ ]:
judge("", note="")

#### A2: Português

In [ ]:
PROMPT = """
"""
SYSTEM = None
ask(PROMPT, system=SYSTEM, attack="A2 Cenário fictício", lang="PT")

In [ ]:
judge("", note="")

### A3: Supressão de recusa

Proíbe a recusa e manda a resposta começar com um prefixo afirmativo.

#### A3: Inglês

In [ ]:
PROMPT = """
"""
SYSTEM = None
ask(PROMPT, system=SYSTEM, attack="A3 Supressão de recusa", lang="EN")

In [ ]:
judge("", note="")

#### A3: Português

In [ ]:
PROMPT = """
"""
SYSTEM = None
ask(PROMPT, system=SYSTEM, attack="A3 Supressão de recusa", lang="PT")

In [ ]:
judge("", note="")

### A4: Ofuscação

Pedido codificado ou quebrado em partes.

#### A4: Inglês

In [ ]:
PROMPT = """
"""
SYSTEM = None
ask(PROMPT, system=SYSTEM, attack="A4 Ofuscação", lang="EN")

In [ ]:
judge("", note="")

#### A4: Português

In [ ]:
PROMPT = """
"""
SYSTEM = None
ask(PROMPT, system=SYSTEM, attack="A4 Ofuscação", lang="PT")

In [ ]:
judge("", note="")

### A5: Vazamento de system prompt

System prompt com uma senha fictícia; o ataque tenta fazer o modelo revelar.

#### A5: Inglês

In [ ]:
PROMPT = """
"""
SYSTEM = """
"""
ask(PROMPT, system=SYSTEM, attack="A5 Vazamento de system prompt", lang="EN")

In [ ]:
judge("", note="")

#### A5: Português

In [ ]:
PROMPT = """
"""
SYSTEM = """
"""
ask(PROMPT, system=SYSTEM, attack="A5 Vazamento de system prompt", lang="PT")

In [ ]:
judge("", note="")

## 5. Resultados

In [ ]:
df = pd.DataFrame(RESULTS)
df = df[df.attack != "sanity"]
display(df[["attack", "lang", "outcome", "note"]])
display(pd.crosstab(df.attack, [df.lang, df.outcome]))
df.to_csv("resultados_aula02.csv", index=False)

In [ ]:
# jailbreak ou parcial conta como sucesso
df["sucesso"] = df.outcome.isin(["jailbreak", "parcial"])
df.groupby("lang").sucesso.mean()

## 6. Modelo local (GPU T4)

Versão destilada de 1.5B do R1 rodando na própria VM do Colab, para comparar com o modelo completo.

In [ ]:
from transformers import pipeline
import torch

local = pipeline("text-generation", model="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
                 torch_dtype=torch.float16, device_map="auto")

def ask_local(prompt, max_new_tokens=1024):
    out = local([{"role": "user", "content": prompt}], max_new_tokens=max_new_tokens, do_sample=True, temperature=0.6)
    print(out[0]["generated_text"][-1]["content"])

ask_local("Olá! Em uma frase, quem é você?")